In [7]:
import pandas as pd
import numpy as np
import pickle
MODELS_CACHE = {}

In [8]:
df = pd.read_parquet("test_data.parquet")

In [11]:
def pred_rv(row, features, target, pitch, p_hand, b_side):
    
    if (pitch, p_hand, b_side) not in MODELS_CACHE:
        with open(f'models/train_only/{pitch}/{pitch}_{p_hand}HP_{b_side}HB.pkl', 'rb') as f:
            MODELS_CACHE[(pitch, p_hand, b_side)] = pickle.load(f)
    model = MODELS_CACHE[(pitch, p_hand, b_side)]

    X_test = row[features]
    predictions = model.predict(X_test) 
    
    return pd.Series(predictions, index=test.index)

In [12]:
rows = []
i=1

# Loop thru every pitch(row) of the test set and call pred_rv to predict each run value
for index, row in df.iterrows():
    if row["pitch_type"] in ["FF", "SI"]:
        pitch = "FB"
    elif row["pitch_type"] in ["FC"]:
        pitch = "FC"
    elif row["pitch_type"] in ["CH", "FS", "FO", "SC"]:
        pitch = "OFF"
    elif row["pitch_type"] in ["SL", "ST", "SV", "CU", "KC", "CS"]:
        pitch = "BB"

    if pitch == "FB" or pitch == "FC":
        features = ["release_speed", "release_height", "release_side", "arm_angle", "release_extension", "ivb", "hb", "spin_rate", "plate_x", "plate_z"]
    else:
        features = ["release_speed", "release_height", "release_side", "arm_angle", "release_extension", "ivb", "hb", "spin_rate", "plate_x", "plate_z", "fb_velo"]    
        
    target = ["rv"]
    
    p_hand = row["p_throws"]
    b_side = row["stand"]

    test = pd.DataFrame([row])
    
    test["xRV"] = float(pred_rv(test, features, target, pitch, p_hand, b_side).iloc[0]) 
    rows.append(test)

    # Print a number for every 100,000 pitches
    if i % 100000 == 0:
        print(i)

    i = i+1

100000
200000
300000
400000
500000


In [13]:
new = pd.concat(rows)

In [14]:
pitch = new.groupby(["pitcher", "pitch_type", "stand"]).agg(
    Pitcher = ("player_name", "first"),
    team = ("pitching_team", "first"),
    Hand = ("p_throws", "first"),
    Count = ("pitch_type", "count"),
    release_speed = ("release_speed", "mean"),
    ivb = ("ivb", "mean"),
    hb = ("hb", "mean"),
    spin_rate = ("spin_rate", "mean"),
    release_height = ("release_height", "mean"),
    release_side = ("release_side", "mean"),
    release_extension = ("release_extension", "mean"),
    arm_angle = ("arm_angle", "mean"),
    fb_velo = ("fb_velo", "mean"),
    xRV = ("xRV", "mean")
).reset_index()

In [15]:
pitch = pitch[pitch["Count"] >= 25]
pitch = pitch[pitch["pitch_type"].isin(["FF", "SI", "FC", "CH", "FS", "FO", "SC", "SL", "ST", "SV", "CU", "KC", "CS"])]
pitch_vs_R = pitch[pitch["stand"] == "R"]
pitch_vs_L = pitch[pitch["stand"] == "L"]

In [16]:
pitch_vs_R.sort_values("xRV").head(10)

,pitcher,pitch_type,stand,Pitcher,team,Hand,Count,release_speed,ivb,hb,spin_rate,release_height,release_side,release_extension,arm_angle,fb_velo,xRV
2765,666277,CH,R,"Soriano, George",STL,R,73,89.736986,-1.589589,11.679452,1543.328767,5.680000,2.352603,6.643836,38.947945,96.515031,-0.026548
2822,666974,FS,R,"Cano, Yennier",BAL,R,62,91.288710,-4.641290,13.016129,1566.870968,5.659839,2.354677,5.930645,21.175806,94.886236,-0.026429
6838,808963,FO,R,"Sasaki, Roki",LAD,R,63,85.576190,-2.552381,2.659048,617.936508,6.260317,1.610159,7.198413,52.433333,97.844110,-0.026356
6840,808963,FS,R,"Sasaki, Roki",LAD,R,198,90.185859,1.484848,7.738788,910.353535,6.258384,1.638485,7.237374,50.968182,97.844110,-0.026001
6074,694819,SL,R,"Misiorowski, Jacob",MIL,R,154,91.970130,2.170130,-3.489351,2537.948052,5.104481,1.938377,7.464286,28.728571,99.827172,-0.025975
3279,669461,CU,R,"Liberatore, Matthew",STL,L,272,79.665074,-17.404412,10.161618,3059.437500,6.623934,-0.625809,6.239338,48.950368,94.029893,-0.025741
4124,676879,CU,R,"Ashby, Aaron",MIL,L,195,83.598462,-17.152615,7.008615,2733.225641,6.478769,-0.407385,5.519487,54.414359,97.620427,-0.024694
5665,690953,FF,R,"Abel, Mick",MIN,R,77,95.172727,16.655065,7.020779,2508.584416,5.485974,1.764935,6.948052,28.524675,94.746667,-0.024003
5349,687330,CH,R,"Kelly, Kevin",TB,R,78,86.015385,-8.075385,12.527692,1880.756410,3.546410,1.680385,6.988462,-0.716667,91.535358,-0.023793
3318,669622,ST,R,"Bender, Anthony",MIA,R,185,83.532973,2.957838,-20.721730,2873.816216,5.137946,2.302270,5.457838,7.704865,96.373184,-0.023296


In [17]:
pitch_vs_L.sort_values("xRV").head(10)

,pitcher,pitch_type,stand,Pitcher,team,Hand,Count,release_speed,ivb,hb,spin_rate,release_height,release_side,release_extension,arm_angle,fb_velo,xRV
3526,670970,SI,L,"Morejon, Adrian",SD,L,212,99.397170,9.007925,-16.021132,2405.174528,5.905047,-2.528538,6.270283,40.155660,99.419533,-0.030003
4786,681911,SL,L,"Vesia, Alex",LAD,L,149,84.300000,3.149799,5.838926,2396.946309,6.173423,0.060872,6.312752,74.633557,92.011436,-0.028314
3223,669373,SL,L,"Skubal, Tarik",LAD,L,27,89.462963,3.577778,3.008889,2219.703704,6.055556,-2.068148,6.429630,45.314815,96.707166,-0.026413
2774,666374,CH,L,"Brash, Matt",SEA,R,32,90.931250,-1.207500,9.255000,1806.218750,5.083750,2.735625,6.487500,27.815625,97.163380,-0.026214
730,606996,ST,L,"Hart, Kyle",SD,L,119,83.243697,0.994286,17.503866,2787.890756,5.554874,-2.525126,6.100000,34.887395,93.865385,-0.025906
348,573204,SL,L,"Thielbar, Caleb",CHC,L,68,87.472059,5.797059,4.891765,2299.500000,6.110441,-0.049559,6.175000,54.245588,93.287059,-0.025542
4189,677053,SL,L,"Nardi, Andrew",MIA,L,32,85.496875,2.122500,5.407500,2356.281250,5.574063,-3.136875,6.606250,39.646875,94.570213,-0.025329
6811,806188,SL,L,"Gibson, Cade",MIA,L,49,86.638776,5.512653,3.240000,2823.816327,5.609592,-1.500816,5.995918,30.534694,92.112575,-0.025221
5767,691799,CU,L,"Taylor, Grant",CWS,R,204,84.021569,-15.634118,-2.024706,2508.117647,5.956716,0.512696,7.089706,61.702451,98.249827,-0.024913
3818,675660,SL,L,"Drohan, Shane",MIL,L,153,86.468627,-0.095686,3.857255,2854.666667,5.479216,-1.637190,6.095425,33.750327,94.018757,-0.024238


In [18]:
from scipy import stats

pitch_vs_R["Pitch+"] = 100 + (-10 * stats.zscore(pitch_vs_R["xRV"]))
pitch_vs_R["Pitch+"] = pitch_vs_R["Pitch+"].round().astype("Int64")

pitch_vs_L["Pitch+"] = 100 + (-10 * stats.zscore(pitch_vs_L["xRV"]))
pitch_vs_L["Pitch+"] = pitch_vs_L["Pitch+"].round().astype("Int64")

/var/folders/lr/9t8mdvqx6_58bf8fjvwdbh3h0000gp/T/ipykernel_60020/1851001220.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pitch_vs_R["Pitch+"] = 100 + (-10 * stats.zscore(pitch_vs_R["xRV"]))
/var/folders/lr/9t8mdvqx6_58bf8fjvwdbh3h0000gp/T/ipykernel_60020/1851001220.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pitch_vs_R["Pitch+"] = pitch_vs_R["Pitch+"].round().astype("Int64")
/var/folders/lr/9t8mdvqx6_58bf8fjvwdbh3h0000gp/T/ipykernel_60020/1851001220.py:6: SettingWithCopyWarning: 
A value is t

In [19]:
pitch_vs_R[pitch_vs_R["pitch_type"] == "ST"].sort_values("Pitch+", ascending=False).head(10)

,pitcher,pitch_type,stand,Pitcher,team,Hand,Count,release_speed,ivb,hb,spin_rate,release_height,release_side,release_extension,arm_angle,fb_velo,xRV,Pitch+
3318,669622,ST,R,"Bender, Anthony",MIA,R,185,83.532973,2.957838,-20.721730,2873.816216,5.137946,2.302270,5.457838,7.704865,96.373184,-0.023296,126
5549,689147,ST,R,"Kerkering, Orion",PHI,R,202,87.016337,-0.705743,-14.432079,2916.544554,5.660050,1.796436,5.997525,29.003465,97.033941,-0.022742,126
1588,643377,ST,R,"Jax, Griffin",TB,R,211,87.796209,-0.687014,-12.584645,3012.502370,5.393175,1.139763,6.323223,40.313270,95.478849,-0.021047,124
3401,669923,ST,R,"Kirby, George",SEA,R,337,87.487537,-3.073709,-11.063501,2384.157270,5.722997,1.636795,6.613947,33.442433,96.498968,-0.020464,123
5758,691725,ST,R,"Painter, Andrew",PHI,R,127,82.957480,2.105197,-16.402205,2626.944882,6.211496,2.641890,6.500000,37.999213,96.163667,-0.020417,123
2061,660271,ST,R,"Ohtani, Shohei",LAD,R,254,85.133465,3.286299,-15.280630,2743.070866,5.589173,2.175315,6.549606,31.962992,97.832500,-0.019272,122
1721,656234,ST,R,"Bird, Jake",WSH,R,171,84.294737,5.797193,-14.661053,2724.444444,5.082456,3.178363,5.829825,13.875439,94.507420,-0.019299,122
4981,683175,ST,R,"Phillips, Connor",CIN,R,140,85.963571,2.274000,-15.369429,2489.571429,5.620357,1.774857,5.997857,35.246429,98.333333,-0.018577,121
4032,676604,ST,R,"Zuber, Tyler",MIA,R,75,83.749333,7.224000,-16.123200,2763.960000,4.744267,2.666000,5.970667,23.908000,95.044444,-0.019056,121
5266,686930,ST,R,"Barnett, Mason",ATH,R,92,83.613043,-4.835217,-15.860870,2697.217391,6.018478,2.330543,6.108696,53.030435,95.018065,-0.019242,121


In [20]:
pitch_vs_L[pitch_vs_L["pitch_type"] == "ST"].sort_values("Pitch+", ascending=False).head(10)

,pitcher,pitch_type,stand,Pitcher,team,Hand,Count,release_speed,ivb,hb,spin_rate,release_height,release_side,release_extension,arm_angle,fb_velo,xRV,Pitch+
730,606996,ST,L,"Hart, Kyle",SD,L,119,83.243697,0.994286,17.503866,2787.890756,5.554874,-2.525126,6.100000,34.887395,93.865385,-0.025906,128
2963,668941,ST,L,"Romero, JoJo",STL,L,157,83.042038,-0.267516,16.196943,2841.038217,5.260318,-2.160510,5.310191,29.187898,93.923795,-0.023943,126
3965,676467,ST,L,"Gordon, Colton",HOU,L,50,81.218000,6.480000,10.123200,2567.900000,5.318200,-2.739000,6.838000,26.206000,91.185000,-0.022473,124
350,573204,ST,L,"Thielbar, Caleb",CHC,L,73,80.367123,-0.969863,14.804384,2487.986301,5.990000,-0.252192,6.124658,49.643836,93.287059,-0.022248,124
6813,806188,ST,L,"Gibson, Cade",MIA,L,119,80.346218,0.285378,21.409412,3179.773109,5.394370,-1.733025,5.809244,24.947059,92.112575,-0.021033,122
2845,667463,ST,L,"King, John",MIA,L,64,81.410938,-1.318125,15.774375,2802.781250,5.477188,-2.023438,5.978125,26.653125,90.428000,-0.020534,122
5910,693855,ST,L,"Seymour, Ian",TB,L,129,81.914729,9.375814,12.165581,2191.705426,5.561938,-1.720543,6.355814,39.789147,90.902852,-0.019712,121
245,548384,ST,L,"Raley, Brooks",PHI,L,95,80.815789,4.345263,17.180211,2917.705263,5.456632,-3.239158,5.883158,28.590526,88.168035,-0.018351,119
4191,677053,ST,L,"Nardi, Andrew",MIA,L,53,82.754717,3.491321,14.981887,2497.867925,5.545472,-3.096792,6.633962,39.492453,94.570213,-0.017334,118
3550,671096,ST,L,"Abbott, Andrew",CIN,L,287,82.264460,-0.878049,13.777422,2853.616725,6.039895,-2.458920,6.536237,51.302439,92.273193,-0.016675,118


In [21]:
print("R: ", len(pitch_vs_R))
print("L: ", len(pitch_vs_L))

R:  2104
L:  2070


In [25]:
pitch_vs_R[pitch_vs_R["team"] == "BOS"].sort_values("Pitch+", ascending=False).head(10)

,pitcher,pitch_type,stand,Pitcher,team,Hand,Count,release_speed,ivb,hb,spin_rate,release_height,release_side,release_extension,arm_angle,fb_velo,xRV,Pitch+
234,547973,FS,R,"Chapman, Aroldis",BOS,L,61,89.868852,2.667541,-5.049836,1167.081967,6.159672,-0.320000,7.347541,54.367213,97.391828,-0.018958,121
236,547973,SI,R,"Chapman, Aroldis",BOS,L,162,98.113580,16.419259,-10.571111,2459.555556,6.198395,-0.311975,7.148148,52.686420,97.391828,-0.018497,121
5473,687941,SL,R,"Gamboa, Alec",BOS,L,36,89.252778,1.700000,3.620000,2190.138889,6.200556,-1.786667,6.644444,57.927778,95.596063,-0.016742,118
3349,669711,ST,R,"Weissert, Greg",BOS,R,81,81.691358,5.813333,-19.080000,2896.333333,4.531605,3.428519,5.839506,6.954321,93.892261,-0.014741,116
5178,686580,ST,R,"Slaten, Justin",BOS,R,61,85.998361,2.268197,-13.925902,2620.606557,5.753279,1.246066,6.791803,42.290164,95.155446,-0.013977,115
238,547973,SL,R,"Chapman, Aroldis",BOS,L,47,85.472340,5.142128,10.598298,2307.255319,6.255106,-0.320213,7.293617,58.465957,97.391828,-0.013587,115
2377,663776,SL,R,"Sandoval, Patrick",BOS,L,90,87.024444,4.089333,7.276000,2597.433333,5.959778,-1.392778,6.178889,52.430000,93.930000,-0.013400,114
3972,676477,SL,R,"Whitlock, Garrett",BOS,R,123,86.002439,-1.733659,-0.434146,1968.634146,5.984228,1.013415,7.213008,44.417073,94.969741,-0.010583,111
480,594027,SI,R,"Guerrero, Tyron",BOS,R,194,99.741237,13.007629,14.061031,2188.319588,6.324948,1.728041,7.141753,43.458763,100.098485,-0.011075,111
482,594027,SL,R,"Guerrero, Tyron",BOS,R,50,91.398000,3.904800,-2.157600,2112.740000,6.221600,1.796400,7.104000,43.422000,100.098485,-0.010645,111


In [26]:
pitch_vs_L[pitch_vs_L["team"] == "BOS"].sort_values("Pitch+", ascending=False).tail(10)

,pitcher,pitch_type,stand,Pitcher,team,Hand,Count,release_speed,ivb,hb,spin_rate,release_height,release_side,release_extension,arm_angle,fb_velo,xRV,Pitch+
235,547973,SI,L,"Chapman, Aroldis",BOS,L,66,97.345455,15.225455,-10.765455,2445.075758,6.148788,-0.430758,7.095455,50.339394,97.391828,0.005109,93
3342,669711,FF,L,"Weissert, Greg",BOS,R,96,93.977083,10.800000,10.135000,2275.770833,5.010208,2.474896,5.826042,16.072917,93.892261,0.005944,92
4325,678394,SI,L,"Bello, Brayan",BOS,R,267,95.242697,5.786966,16.342921,2005.801498,5.635768,1.664831,6.213858,39.796629,93.231250,0.008378,90
6497,701719,FF,L,"Samaniego, Tyler",BOS,L,26,94.207692,14.413846,-3.673846,2193.923077,5.765769,-1.506923,7.046154,39.873077,92.778814,0.009006,89
3344,669711,SI,L,"Weissert, Greg",BOS,R,65,93.707692,1.532308,17.268923,2117.661538,4.780308,2.642000,5.881538,10.264615,93.892261,0.009778,88
385,592454,FF,L,"Kahnle, Tommy",BOS,R,36,93.797222,16.256667,4.790000,2193.611111,5.808889,1.437500,6.750000,42.688889,93.641176,0.010133,88
6438,700842,FF,L,"Rivera, Eduardo",BOS,L,55,95.150909,16.418182,-3.048000,2237.218182,6.327091,-1.739091,6.590909,47.312727,95.321053,0.010441,87
182,543243,SI,L,"Gray, Sonny",BOS,R,168,91.695833,6.800714,11.741429,2315.446429,5.475655,0.654167,6.434524,45.070833,90.714944,0.010976,87
4323,678394,FF,L,"Bello, Brayan",BOS,R,76,95.119737,11.243684,6.680526,2030.197368,5.645658,1.725132,6.171053,40.175000,93.231250,0.016427,81
3483,670245,SI,L,"Watson, Ryan",BOS,R,38,93.594737,7.661053,15.900000,2149.789474,5.512632,1.186053,7.084211,34.681579,92.261790,0.018240,79


In [28]:
pitch_vs_R[pitch_vs_R["Pitcher"] == "Gray, Sonny"].sort_values("Pitch+", ascending=False).tail(10)

,pitcher,pitch_type,stand,Pitcher,team,Hand,Count,release_speed,ivb,hb,spin_rate,release_height,release_side,release_extension,arm_angle,fb_velo,xRV,Pitch+
179,543243,FC,R,"Gray, Sonny",BOS,R,162,88.172840,2.882963,-4.431852,2641.858025,5.561420,0.720309,6.490123,46.880864,90.714944,-0.009581,110
185,543243,ST,R,"Gray, Sonny",BOS,R,169,84.885207,-4.134675,-13.206391,2671.893491,5.579586,0.623018,6.441420,47.816568,90.714944,-0.003166,102
183,543243,SI,R,"Gray, Sonny",BOS,R,217,92.420737,7.458249,11.092535,2374.046083,5.536959,0.594009,6.508756,46.937788,90.714944,0.004234,93
181,543243,FF,R,"Gray, Sonny",BOS,R,54,91.712963,11.744444,-0.677778,2471.277778,5.564074,0.673704,6.461111,46.592593,90.714944,0.005682,91


In [29]:
pitch_vs_R.to_parquet("pitch_vs_R.parquet")
pitch_vs_L.to_parquet("pitch_vs_L.parquet")